<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 2.4: Sequential Logic
**Prev: [Control Flow](2.3_control_flow.ipynb)**<br>
**Next: [FIR Filter](2.5_exercise.ipynb)**

## Motivation
You can't write any meaningful digital logic without state. You can't write any meaningful digital logic without state. You can't write any meaningful digital logic....

Get it? Because without storing intermediate results, you can't get anywhere.

Ok, that bad joke aside, this module will describe how to express common sequential patterns in Chisel. By the end of the module, you should be able to implement and test a shift register in Chisel.

It's important to emphasize that this section will probably not dramatically impress you. Chisel's power is not in new sequential logic patterns, but in the parameterization of a design. Before we demonstrate that capability, we have to learn what these sequential patterns are. Thus, this section will show you that Chisel can do pretty much what Verilog can do - you just need to learn the Chisel syntax.

## Setup
If a cell fails or hangs, see [Troubleshooting](docs/troubleshooting.md).

In [ ]:
interp.load.module(os.pwd / "source" / "load-ivy.sc")

In [ ]:
import chisel3._
import chisel3.Input // avoid a name clash with almond.input.Input
import chisel3.util._
import chisel3.simulator.PeekPokeAPI._

---
# Registers
The basic stateful element in Chisel is the register, denoted `Reg`.
A `Reg` holds its output value until the rising edge of its clock, at which time it takes on the value of its input.
By default, every Chisel `Module` has an implicit clock that is used by every register in the design.
This saves you from always specifying the same clock all over your code.

<span style="color:blue">**Example: Using a Register**</span><br>
The following code block implements a module that takes the input, adds 1 to it, and connects it as the input of a register.
*Note: The implicit clock can be overridden for multi-clock designs. See the appendix for an example.*

In [ ]:
class RegisterModule extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(12.W))
    val out = Output(UInt(12.W))
  })

  val register = Reg(UInt(12.W))
  register := io.in + 1.U
  io.out   := register
}

simulate(new RegisterModule) { c =>
  for (i <- 0 until 100) {
    c.io.in.poke(i.U)
    c.clock.step(1)
    c.io.out.expect((i + 1).U)
  }
}
println("SUCCESS!!")

The register is created by calling `Reg(tpe)`, where `tpe` is a variable that encodes the type of register we want.
In this example, `tpe` is a 12-bit `UInt`.

Look at what the tester above is doing.
Between calls to `poke()` and `expect`, there is a call to `step(1)`.
This tells the test harness to tick the clock once, which will cause the register to pass its input to its output.

Calling `step(n)` will tick the clock `n` times.

The astute observer will notice that previous testers testing combinational logic did not call `step()`. This is because calling `poke()` on an input immediately propagates the updated values through combinational logic. Calling `step()` is only needed to update state elements in sequential logic. 

The code block below will show the SystemVerilog generated by `RegisterModule`.

Note:
* The module has an input for clock (and reset) that you didn't add- this is the implicit clock
* The variable `register` shows up as `reg [11:0]`, as expected
* There is a block sectioned off by `ifdef ENABLE_INITIAL_REG_` / `ifdef RANDOMIZE_REG_INIT` that initializes the register to some random value before simulation starts (this is why a plain `Reg` has no predictable value until you write to it)
* `register` is updated on `posedge clock`

In [ ]:
println(getVerilog(new RegisterModule))

One important note is that Chisel distinguishes between types (like `UInt`) and hardware nodes (like the literal `2.U`, or the output of `myReg`). While
```scala
val myReg = Reg(UInt(2.W))
```
is legal because a Reg needs a data type as a model,
```scala
val myReg = Reg(2.U)
```
is an error because `2.U` is already a hardware node and can't be used as a model.

<span style="color:blue">**Example: RegNext**</span><br>
Chisel has a convenience register object for registers with simple input connections. The previous `Module` can be shortened to the following `Module`. Notice how we didn't need to specify the register bitwidth this time. It gets inferred from the register's output connection, in this case `io.out`.

In [ ]:
class RegNextModule extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(12.W))
    val out = Output(UInt(12.W))
  })

  // register bitwidth is inferred from io.out
  io.out := RegNext(io.in + 1.U)
}

simulate(new RegNextModule) { c =>
  for (i <- 0 until 100) {
    c.io.in.poke(i.U)
    c.clock.step(1)
    c.io.out.expect((i + 1).U)
  }
}
println("SUCCESS!!")

The SystemVerilog looks almost the same as before, though the register name is generated instead of explicity defined.

In [ ]:
println(getVerilog(new RegNextModule))

---
# `RegInit`

The register in `RegisterModule` was initialized to random data for simulation.
Unless otherwised specified, registers do not have a reset value (or a reset).
The way to create a register that resets to a given value is with `RegInit`.

For instance, a 12-bit register initialized to zero can be created with the following.
Both versions below are valid and do the same thing:
```scala
val myReg = RegInit(UInt(12.W), 0.U)
val myReg = RegInit(0.U(12.W))
```

The first version has two arguments.
The first argument is a type node that specified the datatype and its width.
The second argument is a hardware node that specified the reset value, in this case 0.

The second version has one argument.
It is a hardware node that specifies the reset value, but normally `0.U`.

<span style="color:blue">**Example: Initialized Register** </span><br>
The following demonstrates using `RegInit()`, initialized to zero.

In [ ]:
class RegInitModule extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(12.W))
    val out = Output(UInt(12.W))
  })

  val register = RegInit(0.U(12.W))
  register := io.in + 1.U
  io.out   := register
}

println(getVerilog(new RegInitModule))

simulate(new RegInitModule) { c =>
  c.io.out.expect(0.U) // right after the reset: the register holds its reset value, not random junk
  c.io.in.poke(41.U)
  c.clock.step(1)
  c.io.out.expect(42.U)
}
println("SUCCESS!!")

Note that the generated SystemVerilog now has a block that checks `if (reset)` to reset the register to 0.
Also note that this is inside the `always @(posedge clock)` block.
Chisel's implicit reset is active high and synchronous.
The register is still initialized to random junk until reset is applied.

**Important:** `simulate(...)` applies a reset to the module (one clock cycle with `reset` high) before running your test body, like the testers of earlier Chisel versions did.
That is what gives a `RegInit` register its value: after the reset it holds `0` in the test above.
A plain `Reg`, on the other hand, has no reset value, so it holds a *random* value at power-up (ChiselSim randomizes uninitialized registers on purpose, to catch tests that accidentally depend on them) and keeps it until the circuit writes to it.
So if your test needs a register to start at a known value, use `RegInit`.

If you ever need to reset again in the middle of a test, drive the `reset` input yourself like any other input: `c.reset.poke(true.B); c.clock.step(1); c.reset.poke(false.B)`. The reset is synchronous, so it only takes effect on a clock edge.

---
# Control Flow
Registers are very similar to wires in terms of control flow.
They have last connect semantics and can be assigned to conditionally with `when`, `elsewhen`, and `otherwise`.

<span style="color:blue">**Example: Register Control Flow**</span><br>
The following example finds the maximum value in a sequence of inputs using conditional register assignments.
Thanks to the reset applied by `simulate`, `max` starts at 0 in the test below. Had we declared it with a plain `Reg(UInt(10.W))` instead of `RegInit`, the first `expect` would fail, because the register would start out with a random value.

In [ ]:
class FindMax extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(10.W))
    val max = Output(UInt(10.W))
  })

  val max = RegInit(0.U(10.W))
  when(io.in > max) {
    max := io.in
  }
  io.max := max
}

simulate(new FindMax) { c =>
  c.io.max.expect(0.U)
  c.io.in.poke(1.U)
  c.clock.step(1)
  c.io.max.expect(1.U)
  c.io.in.poke(3.U)
  c.clock.step(1)
  c.io.max.expect(3.U)
  c.io.in.poke(2.U)
  c.clock.step(1)
  c.io.max.expect(3.U)
  c.io.in.poke(24.U)
  c.clock.step(1)
  c.io.max.expect(24.U)
}
println("SUCCESS!!")

---
# Other Register Examples

Operations called on a register are performed on the **output** of the register, and the kind of operations depend on the register's type.
That means that you can write
```scala
val reg: UInt = Reg(UInt(4.W))
```
which means the value `reg` is of type `UInt` and you can do things you can normally do with `UInt`s, like `+`, `-`, etc.


You aren't restricted to using `UInt`s with registers, you can use any subclass of the base type `chisel3.Data`. This includes `SInt` for signed integers and a lot of other things.

<span style="color:blue">**Example: Comb Filter**</span><br>
The following example shows a comb filter.

In [ ]:
class Comb extends Module {
  val io = IO(new Bundle {
    val in  = Input(SInt(12.W))
    val out = Output(SInt(12.W))
  })

  val delay: SInt = Reg(SInt(12.W))
  delay  := io.in
  io.out := io.in - delay
}
println(getVerilog(new Comb))

---
# Exercises
<span style="color:red">**Exercise: Shift Register**</span><br>
Given your new-found registering knowledge, build a module that implements a shift register for a LFSR. Specifically:
- Each element is a single bit wide.
- Has a 4-bit output signal.
- Takes a single input bit, which is the next value into the shift register.
- Outputs the parallel output of the shift register, with the most significant bit being the last element of the shift register and the least significant bit being the first element of the shift register. `Cat` may come in handy.
- **The output initializes at `b0001`.**
- Shifts each clock cycle (no enable signal).
- **Note in Chisel, subword assignment IS ILLEGAL**; something like `out(0) := in` will not work.

<img src="images/shifter4.svg" alt="shift register figure" style="width: 450px" />

A basic Module skeleton and a test are provided below. The first register has been provided for you.

In [ ]:
class MyShiftRegister(val init: Int = 1) extends Module {
  val io = IO(new Bundle {
    val in  = Input(Bool())
    val out = Output(UInt(4.W))
  })

  val state = RegInit(UInt(4.W), init.U)

  ???
}

simulate(new MyShiftRegister()) { c =>
  var state = c.init
  for (i <- 0 until 10) {
    // poke in LSB of i (i % 2)
    c.io.in.poke(((i % 2) != 0).B)
    // update expected state
    state = ((state * 2) + (i % 2)) & 0xf
    c.clock.step(1)
    c.io.out.expect(state.U)
  }
}
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>Solution</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  val nextState = (state << 1) | io.in
  state := nextState
  io.out := state
</pre></article></div></section></div>

<span style="color:red">**Exercise: Parameterized Shift Register (Optional)**</span><br>
Write a shift register that is parameterized by its delay (`n`), its initial value (`init`), and also has an enable input signal (`en`).

In [ ]:
// n is the output width (number of delays - 1)
// init state to init
class MyOptionalShiftRegister(val n: Int, val init: BigInt = 1) extends Module {
  val io = IO(new Bundle {
    val en  = Input(Bool())
    val in  = Input(Bool())
    val out = Output(UInt(n.W))
  })

  val state = RegInit(init.U(n.W))

  ???
}

// test different depths
for (i <- Seq(3, 4, 8, 24, 65)) {
  println(s"Testing n=$i")
  simulate(new MyOptionalShiftRegister(n = i)) { c =>
    val inSeq = Seq(0, 1, 1, 1, 0, 1, 1, 0, 0, 1)
    var state = c.init
    var i     = 0
    c.io.en.poke(true.B)
    while (i < 10 * c.n) {
      // poke in repeated inSeq
      val toPoke = inSeq(i % inSeq.length)
      c.io.in.poke((toPoke != 0).B)
      // update expected state
      state = ((state * 2) + toPoke) & BigInt("1" * c.n, 2)
      c.clock.step(1)
      c.io.out.expect(state.U)

      i += 1
    }
  }
}
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-2" />
<label for="check-2"><strong>Solution</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  val nextState = (state << 1) | io.in
  when (io.en) {
    state  := nextState
  }
  io.out := state
</pre></article></div></section></div>

---
# Appendix: Explicit clock and reset
Chisel modules have a default clock and reset that are implicitly used by every register created inside them.
There are times where you want to be able to override this default behavior; perhaps you have a black box that generates a clock or reset signal, or you have a multi-clock design.

Chisel provides constructs for dealing with these cases.
Clocks and resets can be overridden separately or together with `withClock() {}`, `withReset() {}`, and `withClockAndReset() {}`.
The following code blocks will give examples of using these functions.

One thing to be aware of is that the default `reset` is synchronous and of type `Bool` (Chisel also supports asynchronous resets, see `Reset`, `AsyncReset` and `RequireAsyncReset`).
Clocks have their own type in Chisel (`Clock`) and should be declared as such.
*`Bool`s can be converted to `Clock`s by calling `asClock()` on them, but you should be careful that you aren't doing something silly.*

Also note that driving several clocks from a test is more involved than the single-clock `c.clock.step()` used in this module; the examples below therefore only elaborate the hardware.

<span style="color:blue">**Example: Multi-Clock Module**</span><br>
A module with multiple clocks and reset signals.

In [ ]:
// withClock, withReset and withClockAndReset are part of chisel3._, which we already imported
class ClockExamples extends Module {
  val io = IO(new Bundle {
    val in                = Input(UInt(10.W))
    val alternateReset    = Input(Bool())
    val alternateClock    = Input(Clock())
    val outImplicit       = Output(UInt(10.W))
    val outAlternateReset = Output(UInt(10.W))
    val outAlternateClock = Output(UInt(10.W))
    val outAlternateBoth  = Output(UInt(10.W))
  })

  val imp = RegInit(0.U(10.W))
  imp            := io.in
  io.outImplicit := imp

  withReset(io.alternateReset) {
    // everything in this scope with have alternateReset as the reset
    val altRst = RegInit(0.U(10.W))
    altRst               := io.in
    io.outAlternateReset := altRst
  }

  withClock(io.alternateClock) {
    val altClk = RegInit(0.U(10.W))
    altClk               := io.in
    io.outAlternateClock := altClk
  }

  withClockAndReset(io.alternateClock, io.alternateReset) {
    val alt = RegInit(0.U(10.W))
    alt                 := io.in
    io.outAlternateBoth := alt
  }
}

println(getVerilog(new ClockExamples))

<span style="color:blue">**Example: Synchronous and Asynchronous Reset**</span><br>
A register with a reset value (`RegInit`) takes that value when reset is asserted. *When* depends on the
kind of reset:

- a **synchronous** reset (type `Bool`) is sampled at the clock edge, like any other input;
- an **asynchronous** reset (type `AsyncReset`) takes effect immediately, without waiting for the clock.

The `reset` of a `Module` has the abstract type `Reset`; Chisel infers its kind from how it is driven,
and a top-level module gets a synchronous reset. Mixing `RequireAsyncReset` (or `RequireSyncReset`) into
a module fixes the kind instead. The helper below prints only the lines of the SystemVerilog that show
how a register is reset:

In [ ]:
// each `always @(...)` line and the `if (...)` reset check that follows it
def resetLines(sv: String): Unit =
  sv.linesIterator
    .sliding(2)
    .collect { case Seq(a, b) if a.trim.startsWith("always @") => s"$a\n$b" }
    .foreach(println)

class SyncResetReg extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(8.W))
    val out = Output(UInt(8.W))
  })
  io.out := RegNext(io.in, 0.U)
}

// the same register, with the reset kind fixed to asynchronous
class AsyncResetReg extends SyncResetReg with RequireAsyncReset

println("Synchronous reset (the default):")
resetLines(getVerilog(new SyncResetReg))
println("\nAsynchronous reset (RequireAsyncReset):")
resetLines(getVerilog(new AsyncResetReg))

The asynchronous register is sensitive to `posedge reset` as well as `posedge clock`. Most FPGA and ASIC
flows have a preferred kind, so pick one per design and keep it consistent.

Inside a module, `withReset` also accepts an asynchronous reset, so one group of registers can use a
different kind of reset than the rest. `asAsyncReset` turns a `Bool` into an `AsyncReset`:

In [ ]:
class MixedResets extends Module {
  val io = IO(new Bundle {
    val asyncReset = Input(Bool())
    val in         = Input(UInt(8.W))
    val syncOut    = Output(UInt(8.W))
    val asyncOut   = Output(UInt(8.W))
  })
  io.syncOut := RegNext(io.in, 0.U) // the module's synchronous reset
  withReset(io.asyncReset.asAsyncReset) {
    io.asyncOut := RegNext(io.in, 0.U) // reset asynchronously by io.asyncReset
  }
}

resetLines(getVerilog(new MixedResets))

---
# Wrap Up
Great job completing this section!! You've now learned how to create registers and write sequential logic in Chisel, which means you have enough basic building blocks to write real circuits.

The next section will combine everything we've learned into one example! If you need a little more encouragement, just remember these words from an expert Chisel user:

![BobRoss](http://i.qkme.me/3qbd5u.jpg)

---
# You're done!

[Return to the top.](#top)